In [4]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
import pandas as pd

mp_pose = mp.solutions.pose

# Configuration
video_path = './Videos/back_view_good.mov'
video_capture = cv2.VideoCapture(video_path)

fps_raw = video_capture.get(cv2.CAP_PROP_FPS)
frames_per_second = fps_raw if fps_raw > 0 else 30.0

# Parameters
GCT_LIFT_THRESHOLD = 0.015 
STRIKE_LOCKOUT = int(frames_per_second * 0.22) 
history_window = 15

# Tracking Deques
right_ankle_y_history = deque(maxlen=history_window)
left_ankle_y_history = deque(maxlen=history_window)
cadence_history = deque(maxlen=8)
all_step_metrics_storage = []

# State Tracking
leg_is_on_ground = {"Right": False, "Left": False}
ground_y_level = {"Right": 0.0, "Left": 0.0}
last_strike_frame = {"Right": 0, "Left": 0}
previous_strike_time = None
average_cadence = 0
right_step_count, left_step_count = 0, 0
current_status_event = "WAITING"
live_metrics = {"Right": {"whip": 0, "gct": 0}, "Left": {"whip": 0, "gct": 0}}

# Peak Metric Trackers
peak_whip_tracker = {"Right": 0.0, "Left": 0.0}
global_peak_whip = {"Right": 0.0, "Left": 0.0}
global_peak_foot_offset = {"Right": 0.0, "Left": 0.0}

def get_pixel_point(landmarks, index, width, height):
    return np.array([int(landmarks[index].x * width), int(landmarks[index].y * height)])

frame_counter = 0

with mp_pose.Pose(min_detection_confidence=0.7, min_tracking_confidence=0.7) as pose_analyzer:
    while video_capture.isOpened():
        ret, frame = video_capture.read()
        if not ret: break
        
        frame_counter += 1
        if frame_counter % 3 == 0: continue 
            
        display_frame = frame.copy()
        overlay_layer = frame.copy()
        frame_height, frame_width, _ = frame.shape
        results = pose_analyzer.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark
            
            # Full Variable Names for Keypoints
            right_hip = get_pixel_point(landmarks, 24, frame_width, frame_height)
            left_hip = get_pixel_point(landmarks, 23, frame_width, frame_height)
            right_shoulder = get_pixel_point(landmarks, 12, frame_width, frame_height)
            left_shoulder = get_pixel_point(landmarks, 11, frame_width, frame_height)
            right_knee = get_pixel_point(landmarks, 26, frame_width, frame_height)
            left_knee = get_pixel_point(landmarks, 25, frame_width, frame_height)
            right_ankle = get_pixel_point(landmarks, 28, frame_width, frame_height)
            left_ankle = get_pixel_point(landmarks, 27, frame_width, frame_height)
            right_elbow = get_pixel_point(landmarks, 14, frame_width, frame_height)
            left_elbow = get_pixel_point(landmarks, 13, frame_width, frame_height)
            right_wrist = get_pixel_point(landmarks, 16, frame_width, frame_height)
            left_wrist = get_pixel_point(landmarks, 15, frame_width, frame_height)

            # torso and cross-lines
            torso_points = np.array([right_shoulder, left_shoulder, left_hip, right_hip], np.int32)
            cv2.fillPoly(overlay_layer, [torso_points], (0, 255, 0)) 
            cv2.addWeighted(overlay_layer, 0.15, display_frame, 0.85, 0, display_frame)
            cv2.polylines(display_frame, [torso_points], True, (255, 255, 255), 2)
            cv2.line(display_frame, tuple(right_shoulder), tuple(left_hip), (255, 255, 255), 1)
            cv2.line(display_frame, tuple(left_shoulder), tuple(right_hip), (255, 255, 255), 1)
            
            # connection lines
            for h_p, k_p, a_p, color in [(right_hip, right_knee, right_ankle, (0, 255, 0)), 
                                         (left_hip, left_knee, left_ankle, (0, 255, 255))]:
                cv2.line(display_frame, tuple(h_p), tuple(k_p), color, 3)
                cv2.line(display_frame, tuple(k_p), tuple(a_p), color, 3)
            for s_p, e_p, w_p in [(right_shoulder, right_elbow, right_wrist), 
                                  (left_shoulder, left_elbow, left_wrist)]:
                cv2.line(display_frame, tuple(s_p), tuple(e_p), (255, 165, 0), 3)
                cv2.line(display_frame, tuple(e_p), tuple(w_p), (255, 165, 0), 3)

            # step logic
            current_frame_position = video_capture.get(cv2.CAP_PROP_POS_FRAMES)
            right_ankle_y_history.append(landmarks[28].y)
            left_ankle_y_history.append(landmarks[27].y)
            
            for side, history_queue, ankle_index, knee_index in [("Right", right_ankle_y_history, 28, 26), 
                                                               ("Left", left_ankle_y_history, 27, 25)]:
                current_y_coordinate = landmarks[ankle_index].y
                hip_index = 24 if side == "Right" else 23
                
                # offset angle
                vector_thigh = np.array([landmarks[knee_index].x - landmarks[hip_index].x, 
                                         landmarks[knee_index].y - landmarks[hip_index].y])
                vector_shin = np.array([landmarks[ankle_index].x - landmarks[knee_index].x, 
                                        landmarks[ankle_index].y - landmarks[knee_index].y])
                
                unit_thigh = vector_thigh / np.linalg.norm(vector_thigh)
                unit_shin = vector_shin / np.linalg.norm(vector_shin)
                
                current_offset_angle = np.degrees(np.arccos(np.clip(np.dot(unit_thigh, unit_shin), -1.0, 1.0)))
                
                if current_offset_angle > global_peak_foot_offset[side]:
                    global_peak_foot_offset[side] = current_offset_angle

                if not leg_is_on_ground[side]:
                    # heel whip calculation
                    instant_whip = np.degrees(np.arctan2(abs(landmarks[ankle_index].x - landmarks[knee_index].x), 
                                                         abs(landmarks[ankle_index].y - landmarks[knee_index].y)))
                    if instant_whip > peak_whip_tracker[side]: 
                        peak_whip_tracker[side] = instant_whip
                    if instant_whip > global_peak_whip[side]: 
                        global_peak_whip[side] = instant_whip

                    # detection of strike
                    if len(history_queue) >= 10 and current_y_coordinate >= max(list(history_queue)[:-1]) and (current_frame_position - last_strike_frame[side]) > STRIKE_LOCKOUT:
                        leg_is_on_ground[side] = True
                        ground_y_level[side] = current_y_coordinate
                        last_strike_frame[side] = current_frame_position
                        if previous_strike_time is not None:
                            cadence_history.append((60 * frames_per_second) / (current_frame_position - previous_strike_time))
                            average_cadence = np.mean(cadence_history)
                        previous_strike_time = current_frame_position
                        
                        if side == "Right": right_step_count += 1
                        else: left_step_count += 1
                        current_status_event = f"{side.upper()} STRIKE"
                        
                        all_step_metrics_storage.append({
                            'side': side, 
                            'start_frame': current_frame_position, 
                            'done': False,
                            'cadence': average_cadence, 
                            'heel_whip_val': peak_whip_tracker[side],
                            'foot_offset_val': global_peak_foot_offset[side],
                            'hip_drop_val': np.degrees(np.arctan2(landmarks[24].y - landmarks[23].y, 
                                                                 landmarks[24].x - landmarks[23].x)),
                            'shoulder_drop_val': np.degrees(np.arctan2(landmarks[12].y - landmarks[11].y, 
                                                                     landmarks[12].x - landmarks[11].x))
                        })
                        peak_whip_tracker[side] = 0.0

                elif leg_is_on_ground[side]:
                    if (ground_y_level[side] - current_y_coordinate) > GCT_LIFT_THRESHOLD:
                        for entry in reversed(all_step_metrics_storage):
                            if entry['side'] == side and not entry['done']:
                                entry['gct'] = ((current_frame_position - entry['start_frame']) / frames_per_second) * 1000
                                live_metrics[side]["gct"] = entry['gct']
                                entry['done'] = True
                                leg_is_on_ground[side] = False
                                current_status_event = f"{side.upper()} PUSH-OFF"
                                break

            # visuals
            cv2.rectangle(display_frame, (0, 0), (280, 100), (20, 20, 20), -1)
            cv2.putText(display_frame, f"STEPS: {right_step_count + left_step_count}", (15, 35), 1, 1.8, (255, 255, 255), 2)
            cv2.putText(display_frame, f"CADENCE: {int(average_cadence)}", (15, 65), 1, 1.2, (0, 255, 0), 2)
            cv2.putText(display_frame, f"STATUS: {current_status_event}", (15, 95), 1, 1.0, (0, 255, 255), 1)

            for side_name, position, grounded in [("LEFT", (10, frame_height-20), leg_is_on_ground["Left"]), 
                                                  ("RIGHT", (frame_width-135, frame_height-20), leg_is_on_ground["Right"])]:
                status_color = (0, 255, 0) if grounded else (0, 0, 255)
                cv2.rectangle(display_frame, (position[0]-10, position[1]-60), (position[0]+125, position[1]+10), (0,0,0), -1)
                cv2.putText(display_frame, side_name, (position[0], position[1]-40), 1, 1.2, status_color, 2)
                cv2.putText(display_frame, "CONTACT" if grounded else "FLIGHT", (position[0], position[1]-20), 1, 0.9, (255,255,255), 1)

            cv2.imshow('Running Analysis', display_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'): break

video_capture.release()
cv2.destroyAllWindows()

I0000 00:00:1769201265.413945 3560260 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3
W0000 00:00:1769201265.473554 3728836 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769201265.484167 3728836 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [5]:
import csv
import os

file_path = 'dataset_backview.csv'
fieldnames = [
    'rep_number', 'frame_index', 'file_name','side', 'cadence_val',
    'heel_whip_value (deg)', 'foot_offset_val (deg)', 'hip_drop_value (deg)', 'shoulder_drop_val (deg)',
    'cadence_score', 'heel_whip_score', 'foot_offset_score', 'hip_drop_score', 'shoulder_drop_score'
]

def get_backview_score(metric, val):
    val = abs(val)
    
    if metric == "cadence":
        if val >= 175: return 3
        if val >= 165: return 2
        if val >= 150: return 1
        return 0
    if metric == "heel_whip":
        if val < 7.0: return 3
        if 7.0 <= val <= 10.0: return 2
        if 10.0 <= val <= 15.0: return 1
        return 0
    if metric == "foot_offset":
        if val < 6.0: return 3
        if 6.0 <= val <= 9.0: return 2
        if 9.0 <= val <= 13.0: return 1
        return 0
    if metric == "hip_drop":
        if val < 5.0: return 3
        if 5.0 <= val <= 7.0: return 2
        if 7.0 <= val <= 10.0: return 1
        return 0
    if metric == "shoulder_drop":
        if val < 7.0: return 3
        if 7.0 <= val <= 10.0: return 2
        if 10.0 <= val <= 15.0: return 1
        return 0
    return 0

if not os.path.exists(file_path):
    with open(file_path, 'w', newline='') as f:
        csv.DictWriter(f, fieldnames=fieldnames).writeheader()

if 'all_step_metrics_storage' in locals() and all_step_metrics_storage:
    with open(file_path, 'a', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        for i, m in enumerate(all_step_metrics_storage, start=1):
            if not m.get('done'): continue
          
            cad = m.get('cadence', average_cadence)
            whip = m.get('heel_whip_val', 0)
            f_off = m.get('foot_offset_val', 0)
            h_drop = m.get('hip_drop_val', 0)
            s_drop = m.get('shoulder_drop_val', 0)

            writer.writerow({
                'rep_number': i,
                'frame_index': int(m['start_frame']),
                'file_name': video_path,
                'side': m['side'],
                'cadence_val': round(cad, 1),
                'heel_whip_value (deg)': round(whip, 1),
                'foot_offset_val (deg)': round(f_off, 1),
                'hip_drop_value (deg)': round(h_drop, 1),
                'shoulder_drop_val (deg)': round(s_drop, 1),
                'cadence_score': get_backview_score("cadence", cad),
                'heel_whip_score': get_backview_score("heel_whip", whip),
                'foot_offset_score': get_backview_score("foot_offset", f_off),
                'hip_drop_score': get_backview_score("hip_drop", h_drop),
                'shoulder_drop_score': get_backview_score("shoulder_drop", s_drop)
            })
    print(f"Data saved to {file_path}")

Data saved to dataset_backview.csv


In [6]:
if all_step_metrics_storage:
    final_results = []
    # Metrics configuration for the loop
    metrics_config = [
        ('cadence', 'CADENCE', 'cadence'),
        ('heel_whip_val', 'HEEL WHIP', 'heel_whip'),
        ('foot_offset_val', 'FOOT OFFSET', 'foot_offset'),
        ('hip_drop_val', 'HIP DROP', 'hip_drop'),
        ('shoulder_drop_val', 'SHOULDER DROP', 'shoulder_drop')
    ]
    
    for key, label, score_key in metrics_config:
        valid_vals = [abs(s[key]) if key != 'cadence' else s[key] 
                      for s in all_step_metrics_storage if key in s and s.get('done')]
        
        avg = np.mean(valid_vals) if valid_vals else 0
        score = get_backview_score(score_key, avg)
        final_results.append({"METRIC": label, "AVG VALUE": round(avg, 1), "SCORE": score})

    summary_df = pd.DataFrame(final_results)
    print("\n" + "═"*45)
    print("      POSTERIOR GAIT ANALYSIS SUMMARY")
    print("═"*45)
    print(summary_df.to_string(index=False))
    print("─" * 45)
    print(f"TOTAL SCORE: {summary_df['SCORE'].sum()} / 15")
    print("═"*45)
else:
    print("No gait data found in storage.")


═════════════════════════════════════════════
      POSTERIOR GAIT ANALYSIS SUMMARY
═════════════════════════════════════════════
       METRIC  AVG VALUE  SCORE
      CADENCE      167.5      2
    HEEL WHIP        7.5      2
  FOOT OFFSET       11.2      1
     HIP DROP        1.7      3
SHOULDER DROP       11.0      1
─────────────────────────────────────────────
TOTAL SCORE: 9 / 15
═════════════════════════════════════════════
